In [1]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

np.random.seed(42)
n = 500
X = pd.DataFrame({
    "f1": np.random.normal(0, 1, n),
    "f2": np.random.normal(0, 1, n)
})

y = ((X["f1"] + X["f2"] + np.random.normal(0, 0.5, n)) > 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [2]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train,y_train)
pred_dummy = dummy.predict(X_test)
acc_dummy = accuracy_score(y_test,pred_dummy)

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train,y_train)
pred_rf = rf.predict(X_test)
acc_rf = accuracy_score(y_test, pred_rf)


print("Dummy accuracy:", acc_dummy)
print("Random Forest accuracy:", acc_rf)

Dummy accuracy: 0.47333333333333333
Random Forest accuracy: 0.8533333333333334


In [3]:
import duckdb
import pandas as pd
import numpy as np

duckdb.sql("CREATE VIEW duolingo_flagship AS SELECT * FROM read_csv_auto('../../data/duolingo_flagship_v4.csv')")
df = duckdb.sql("SELECT * FROM duolingo_flagship").df()

# Load the split saved in part 11 instead of re-deriving it here.
split = pd.read_csv("../../data/split_users.csv")
cv_users = set(split.loc[split["split"] == "cv", "user_id"])
df_cv = df[df["user_id"].isin(cv_users)]

df_cv.shape

(14438, 17)

In [4]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
feature_cols = ["lag_days", "history_seen", "history_correct", "history_accuracy", "lag_days_log"]
X_cv = df_cv[feature_cols]
y_cv = df_cv["p_recall"]

# Part 12'deki dummy baseline'larla (mean RMSE 0.276 / median 0.295) karşılaştırmak için.
# random_state sabit, yoksa her çalıştırmada skor biraz oynuyor.
rf_reg = RandomForestRegressor(random_state=42)
rf_reg.fit(X_cv,y_cv)
pred_rf = rf_reg.predict(X_cv)
rmse_rf = np.sqrt(mean_squared_error(y_cv,pred_rf))

print("Random Forest RMSE:", rmse_rf)

Random Forest RMSE: 0.15469765001513097


## Notes

Random Forest (default params, `random_state=42`) beat both dummy baselines by a wide margin: RMSE 0.155 vs 0.276 (mean) / 0.295 (median). Real signal exists in lag_days, history_seen, history_correct, history_accuracy, lag_days_log.

Caveat: this is in-sample (predicted on the same data it was trained on), not a real holdout evaluation. Random Forest with no depth limit can partially memorize training data, so 0.155 is likely optimistic. This was just a signal check, not real model evaluation, proper CV-based evaluation with GroupKFold is later.

All three numbers now come from the split saved in part 11 (`data/split_users.csv`), so they're comparable to each other. The earlier figures (0.157 vs 0.279/0.300) were measured on a split that row reordering later changed.